In [1]:
import os
import sys
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from dotenv import load_dotenv
import json

sys.path.append(os.path.abspath("../.."))
from scraper import fetch_website_contents, fetch_website_links

In [2]:
load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')

if groq_api_key:
    print(f"GROQ API Key exists and begins {groq_api_key[:4]}")
else:
    print("GROQ API Key not set (and this is optional)")

GROQ API Key exists and begins gsk_


In [3]:
groq_url = "https://api.groq.com/openai/v1"

groq = OpenAI(
    base_url=groq_url,
    api_key=groq_api_key
)

MODEL = "openai/gpt-oss-120b"

In [4]:
links = fetch_website_links("https://ollama.com")
links

['/',
 '/search',
 '/docs',
 '/pricing',
 '/signin',
 '/download',
 '/search',
 '/download',
 '/docs',
 '/pricing',
 '/signin',
 'https://docs.ollama.com/integrations/openclaw',
 '/download/windows',
 '/download',
 'https://docs.ollama.com/integrations',
 '/signup',
 '/upgrade?plan=pro&interval=year',
 '/upgrade?plan=pro',
 '/upgrade?plan=max',
 '/download',
 '/download',
 '/blog',
 'https://docs.ollama.com',
 'https://github.com/ollama/ollama',
 'https://discord.com/invite/ollama',
 'https://twitter.com/ollama',
 'mailto:hello@ollama.com',
 '/privacy',
 '/terms',
 '/blog',
 '/download',
 'https://docs.ollama.com',
 'https://github.com/ollama/ollama',
 'https://discord.com/invite/ollama',
 'https://twitter.com/ollama',
 'https://lu.ma/ollama',
 '/privacy',
 '/terms']

## First we are using llama3.2:3b model to figure out which links are relevant for broucher.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the working URL starting with https only in JSON format as per the system prompt instructions.,
Do not include Terms of Service, Privacy, email links, not found, empty links.

Include full https links only, do not include relative links or links that are not working.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://ollama.com"))


Here is the list of links on the website https://ollama.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the working URL starting with https only in JSON format as per the system prompt instructions.,
Do not include Terms of Service, Privacy, email links, not found, empty links.

Include full https links only, do not include relative links or links that are not working.

Links (some might be relative links):

/
/search
/docs
/pricing
/signin
/download
/search
/download
/docs
/pricing
/signin
https://docs.ollama.com/integrations/openclaw
/download/windows
/download
https://docs.ollama.com/integrations
/signup
/upgrade?plan=pro&interval=year
/upgrade?plan=pro
/upgrade?plan=max
/download
/download
/blog
https://docs.ollama.com
https://github.com/ollama/ollama
https://discord.com/invite/ollama
https://twitter.com/ollama
mailto:hello@ollama.com
/privacy
/terms
/blog
/download
https://docs.ollama.com
https://github.com/ollama/ollama
h

In [8]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [9]:
select_relevant_links("https://huggingface.co/")

Selecting relevant links for https://huggingface.co/ by calling openai/gpt-oss-120b
Found 7 relevant links


{'links': [{'type': 'careers page',
   'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'zhihu', 'url': 'https://www.zhihu.com/org/huggingface'},
  {'type': 'product page', 'url': 'https://endpoints.huggingface.co'}]}

In [10]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        if link['url'].startswith("https://"):
            result += f"\n\n### Link: {link['type']}\n"
            result += fetch_website_contents(link["url"])
    return result

In [11]:
print(fetch_page_and_all_relevant_links("https://huggingface.co/"))

Selecting relevant links for https://huggingface.co/ by calling openai/gpt-oss-120b
Found 9 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
bytedance-research/Lance
Updated
1 day ago
•
739
•
592
openbmb/MiniCPM-V-4.6
Updated
3 days ago
•
196k
•
891
Supertone/supertonic-3
Updated
4 days ago
•
35k
•
545
SulphurAI/Sulphur-2-base
Updated
about 5 hours ago
•
1.2M
•
1.24k
unsloth/Qwen3.6-27B-MTP-GGUF
Updated
2 days ago
•
478k
•

In [12]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [13]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:7_000] # Truncate if more than 7,000 characters
    return user_prompt

In [14]:
get_brochure_user_prompt("edwarddonner", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling openai/gpt-oss-120b
Found 9 relevant links


'\nYou are looking at a company called: edwarddonner\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHome - Edward Donner\n\nSkip to content\nHome\nAI Curriculum\nProficient AI Engineer\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,\nacquired in 20

In [15]:
def create_brochure(company_name, url):
    response = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [16]:
create_brochure("HuggingFace", "https://huggingface.co/")

Selecting relevant links for https://huggingface.co/ by calling openai/gpt-oss-120b
Found 7 relevant links


**Hugging Face – The AI Community Building the Future**  
*Your gateway to open‑source machine‑learning, collaboration, and enterprise‑grade AI.*

---

## 1. Who We Are  

Hugging Face is the world’s leading platform where researchers, engineers, and creators come together to **create, discover, and deploy** machine‑learning models, datasets, and applications. With more than **2 million models**, **500 k+ datasets**, and **1 million+ community‑built apps (Spaces)**, we power everything from cutting‑edge research to production‑grade AI services.

---

## 2. Core Products & Services  

| Offering | What It Does | Why It Matters |
|----------|--------------|----------------|
| **Models Hub** | Host, share, and fine‑tune pretrained models (e.g., Llama, Gemma, Qwen). | Accelerates development – you can start from state‑of‑the‑art checkpoints in seconds. |
| **Datasets Hub** | Curated, searchable collections of training and evaluation data. | Saves weeks of data‑collection effort and ensures reproducibility. |
| **Spaces** | Interactive, deploy‑ready ML apps built with Gradio, Streamlit, or custom front‑ends. | Turn a model into a live demo or product with a single click. |
| **Inference Endpoints & Providers** | Scalable, low‑latency hosted inference (cloud‑agnostic, private VPC). | Guarantees reliable performance for enterprise workloads. |
| **Storage Buckets** | Secure, versioned object storage for large model artefacts. | Keeps assets safe, searchable, and ready for CI/CD pipelines. |
| **Enterprise Solutions** | Dedicated support, custom SLAs, on‑prem deployment, and compliance tooling. | Enables Fortune‑500, fintech, health‑tech and other regulated customers to adopt AI responsibly. |
| **Hugging Face PRO & Enterprise Support** | Priority assistance, consulting, and training for teams of any size. | Helps teams move from prototype to production faster. |

---

## 3. Community & Culture  

- **Open‑Source First** – All core libraries (🤗 Transformers, Datasets, Tokenizers, Accelerate) are free, permissively licensed, and maintained by a global community of contributors.  
- **Collaboration‑Driven** – The Hub lets anyone publish models or datasets with just a few clicks, fostering rapid iteration and peer review.  
- **Inclusive & Transparent** – Public forums, Discord, and a vibrant **Hugging Face Forum** host discussions ranging from beginner questions to advanced research debates.  
- **Innovation at Scale** – Daily Papers, blog posts, and community challenges keep the ecosystem at the frontier of AI research.  
- **Developer‑Friendly** – Tight integration with GitHub (issues, actions, codespaces) and tools like GitHub Copilot, making AI development feel native to software engineering workflows.  

> *“The platform where the machine learning community collaborates on models, datasets, and applications.”* – Hugging Face tagline

---

## 4. Customers & Impact  

While the Hub is public, Hugging Face’s **Enterprise** arm powers AI for:

- **Tech giants** building large‑scale LLM services.  
- **Financial institutions** applying NLP for compliance and risk analysis.  
- **Healthcare providers** leveraging medical imaging models with strict privacy guarantees.  
- **Manufacturers & robotics firms** using vision‑and‑language models for quality control.  

These customers rely on **Inference Endpoints**, **Private Buckets**, and **Enterprise Support** to meet production SLAs and regulatory requirements.

---

## 5. Careers – Join the AI Movement  

Hugging Face is constantly expanding its team across:

- **Research & ML Engineering** – Build next‑gen models, improve training efficiency, explore multimodal AI.  
- **Product & Design** – Shape intuitive experiences for the Hub, Spaces, and enterprise dashboards.  
- **Infrastructure & DevOps** – Scale the backend that serves billions of requests daily.  
- **Customer Success & Enterprise Sales** – Partner with global brands to embed responsible AI.  
- **Community & Education** – Run workshops, curate tutorials, and grow the open‑source ecosystem.  

Visit the **Hugging Face Careers** page for current openings and apply to a company where **collaboration, openness, and impact** are core values.

---

## 6. Get Involved  

- **Explore the Hub** – Browse 2 M+ models or 500 k+ datasets and start experimenting instantly.  
- **Launch a Space** – Deploy a demo app in minutes; showcase your work to the world.  
- **Contribute** – Submit pull requests to 🤗 Transformers, report bugs, or share a new dataset.  
- **Join the Conversation** – Participate in the Hugging Face Forum, Discord, or weekly community calls.  
- **Stay Updated** – Follow the status page for real‑time service health, read the Daily Papers blog, and subscribe to the newsletter for the latest research highlights.

---

### Quick Links  

- **Hub & Docs** – https://huggingface.co  
- **Enterprise** – https://huggingface.co/enterprise  
- **Careers** – https://huggingface.co/careers  
- **Forum** – https://discuss.huggingface.co  
- **Status** – https://status.huggingface.co  

---

**Hugging Face** – *Where AI is built together.*  



---  



*Prepared for prospective customers, investors, and talent seeking to be part of the most collaborative AI ecosystem.*

In [17]:
def stream_brochure(url, company_name):
    stream = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [19]:
stream_brochure("https://huggingface.co", "Hugging Face")

Selecting relevant links for https://huggingface.co by calling openai/gpt-oss-120b
Found 7 relevant links


**Hugging Face – The AI Community Building the Future**  
*Collaboration platform for the machine‑learning ecosystem*  

---  

### 🌍 Who We Are  
Hugging Face is the central hub where millions of AI researchers, engineers, and creators come together to **share, discover, and experiment** with open‑source machine‑learning models, datasets, and applications. Our mission is to enable an **open and ethical AI future** by empowering the next generation of talent with powerful tools, a vibrant community, and transparent best‑practice resources.  

**Key facts**  

| Metric | Detail |
|--------|--------|
| **Community size** | >2 M publicly hosted models, >500 k datasets, >1 M AI applications (Spaces) |
| **Core offerings** | Models, Datasets, Spaces (interactive apps), Buckets (storage), Inference Endpoints, HuggingChat |
| **Open‑source impact** | Hosts the most‑used ML libraries (🤗 Transformers, 🤗 Datasets, 🤗 Tokenizers, etc.) |
| **Geography** | Global, with contributors from every continent |
| **Values** | Openness, collaboration, inclusivity, ethical AI, rapid innovation |  

---  

### 🚀 Platform & Products  

| Product | What It Does | Who Benefits |
|---------|--------------|--------------|
| **Model Hub** | Browse, host, and version‑control >2 M models (LLMs, vision, audio, RL, diffusion, etc.) | Researchers, startups, enterprise AI teams |
| **Datasets Hub** | Over 500 k curated datasets, searchable by task, language, size | Data scientists, academia, product teams |
| **Spaces** | Deploy interactive AI demos (e.g., 3‑D generation, multilingual TTS, video‑from‑image) with zero‑code or custom code | Developers, educators, community builders |
| **Inference Endpoints** | Managed, low‑latency serving of models in the cloud or on‑prem | Enterprises needing production‑grade AI |
| **Buckets** | Scalable storage for large model weights and dataset artifacts | Teams handling massive AI assets |
| **HuggingChat** | Conversational AI interface powered by open‑source LLMs | End‑users, developers exploring chat capabilities |
| **Enterprise & PRO plans** | Dedicated support, SLAs, private repos, advanced security | Large organizations, regulated industries |

---  

### 🤝 Community & Culture  

* **Collaboration‑first** – Every model, dataset, or app can be forked, improved, and re‑published, mirroring the open‑source spirit of GitHub.  
* **Ethical AI** – We actively promote responsible AI development, transparency, and bias mitigation (see our blog series on ethics, RLHF, and bias‑free audio).  
* **Learning Hub** – “Learn”, “Daily Papers”, and “Discord/Forum” channels keep the community up‑to‑date on breakthroughs.  
* **Inclusivity** – Brand assets feature a palette of friendly colors (#FFD21E, #FF9D00, #6B7280) and open‑source logos to signal an approachable, diverse environment.  
* **Rapid Innovation** – Our science team pushes the frontier of transformer efficiency, KV‑caching, and low‑resource training (e.g., “Training‑Free Reasoning at 88.89%”).  

---  

### 📈 Customers & Real‑World Impact  

While Hugging Face’s platform is public, it powers a broad spectrum of customers:

| Sector | Typical Use Cases |
|--------|-------------------|
| **Tech & SaaS** | Deploying LLM APIs, chatbots, recommendation engines |
| **Healthcare** | Fine‑tuning biomedical language models on proprietary data |
| **Finance** | Sentiment analysis, fraud detection, risk modeling |
| **Media & Entertainment** | Text‑to‑video generation, multilingual TTS, content moderation |
| **Robotics** | Sim‑to‑real transfer with open‑source RL datasets (e.g., LeRobot Humanoid) |
| **Education** | Interactive notebooks, model‑exploration labs for students |

Many of the most‑downloaded models (e.g., MiniCPM‑V‑4.6, Qwen 3.6‑27B) are integrated into commercial products ranging from virtual assistants to autonomous agents.

---  

### 👩‍💼 Careers – Join the AI Revolution  

Hugging Face is constantly expanding its talent pool. Highlights for prospective hires:

* **Roles** – Research scientists, ML engineers, product managers, data engineers, community managers, UX designers, and security specialists.  
* **Work style** – Remote‑first, flexible hours, and a culture that values **autonomy, mentorship, and continuous learning**.  
* **Impact** – Contribute to tools that will be used by **millions** of developers worldwide and shape the ethical standards of future AI.  
* **Benefits** – Competitive compensation, equity, learning budget, open‑source contribution time, and access to internal research labs.  

Visit the **Careers** section on the website for current openings and application details.  

---  

### 📞 Get Started  

* **Explore models & datasets** – https://huggingface.co/models & https://huggingface.co/datasets  
* **Try an AI app** – https://huggingface.co/spaces (search “Pixal3D”, “Supertonic 3 TTS”, etc.)  
* **Read the latest insights** – https://huggingface.co/blog (topics: ethics, diffusion, RLHF, KV‑caching)  
* **Join the conversation** – Discord, Forum, and GitHub communities are open to all skill levels.  

**Let’s build the future of AI together.**  

---  

*Prepared for prospective customers, investors, and talent seekers.*

### Now we are going to make UI for this website broucher generator using Gradio.

#### This is week-2, day-2 exercise :)

In [20]:
import gradio as gr

/home/jaimin.rana@agshealth.com/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
def stream_brochure_for_gr(url, company_name):
    stream = groq.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
message_input = gr.Textbox(label="Your website link:", info="Enter the URL of the website for which you want to generate a brochure", lines=1)
message_input2 = gr.Textbox(label="Your company name:", info="Enter the name of the company", lines=1)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure_for_gr,
    title="Website brochure generator", 
    inputs=[message_input, message_input2], 
    outputs=[message_output], 
    examples=[
        ["https://huggingface.co", "Hugging Face"],
        ["https://ollama.com", "Ollama"],
        ["https://udemy.com", "Udemy"],
        ["https://edwarddonner.com", "Edward Donner"]
        ], 
    flagging_mode="never"
    )

view.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Selecting relevant links for https://udemy.com by calling openai/gpt-oss-120b
Found 1 relevant links
Selecting relevant links for https://ollama.com by calling openai/gpt-oss-120b
Found 6 relevant links
